# 12 · Qualidade · Checagens do pipeline
Roda checagens sobre a silver e a gold e **guarda o resultado de cada execução** em `gold.dq_resultados`. O histórico permite ver quando um problema apareceu, e o alerta do Caio (ou o dashboard) pode ler a última execução.

| Tipo de checagem | Exemplo | Esperado |
|---|---|---|
| **Sujeira plantada** (D17) | vendas do ERP sem SKU | o número plantado (400, 401, 801): prova que a checagem **encontra** o problema |
| **Regras do contrato** do site | todo pedido tem exatamente um `purchase` | 0 |
| **Reconciliação** entre camadas | receita da gold = receita da fato | 0 diferenças |
| **Informativa** | eventos reenviados pela exportação | sem valor fixo (`INFO`) |

`status`: `OK` quando o encontrado é igual ao esperado; `FALHA` quando difere; `INFO` quando não há valor esperado.

**Por que a sujeira plantada tem esperado diferente de zero:** uma checagem que sempre dá zero pode estar quebrada. Com o problema plantado de propósito, o resultado certo é encontrar exatamente o que foi plantado.

**Equivalente no BigQuery:** a mesma tabela num dataset de monitoramento; `COUNT_IF` → `COUNTIF`; `date_format(ts, 'yyyyMMddHHmmss')` → `FORMAT_TIMESTAMP('%Y%m%d%H%M%S', ts)`. No GCP, o Dataform tem *assertions*, que fazem o mesmo papel dentro do pipeline.

In [0]:
CREATE TABLE IF NOT EXISTS horizonte.gold.dq_resultados (
  execucao_id  STRING,
  executado_em TIMESTAMP,
  camada       STRING,
  tabela       STRING,
  checagem     STRING,
  descricao    STRING,
  esperado     BIGINT,
  encontrado   BIGINT,
  status       STRING
);

### Execução
20 checagens, uma linha por checagem. Cada execução acrescenta um novo bloco de linhas (`INSERT INTO`), sem apagar as anteriores.

In [0]:
INSERT INTO horizonte.gold.dq_resultados
WITH checagens AS (
  SELECT 'silver' AS camada, 'erp_vendas' AS tabela, 'erp_sku_nulo' AS checagem,
         'Sujeira plantada: vendas sem SKU (D17)' AS descricao,
         CAST(400 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(sku_nulo) FROM horizonte.silver.erp_vendas) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'erp_vendas' AS tabela, 'erp_quantidade_negativa' AS checagem,
         'Sujeira plantada: quantidade negativa (D17)' AS descricao,
         CAST(401 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(quantidade_negativa) FROM horizonte.silver.erp_vendas) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'erp_vendas' AS tabela, 'erp_duplicada' AS checagem,
         'Sujeira plantada: linhas repetidas (D17)' AS descricao,
         CAST(801 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(duplicada) FROM horizonte.silver.erp_vendas) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'erp_vendas' AS tabela, 'erp_sku_sem_cadastro' AS checagem,
         'SKU preenchido que não existe no cadastro' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(sku_sem_cadastro) FROM horizonte.silver.erp_vendas) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_pedidos' AS tabela, 'site_status_invalido' AS checagem,
         'Status fora de FINALIZADO e CANCELADO (contrato)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(status_invalido) FROM horizonte.silver.site_pedidos) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_pedidos' AS tabela, 'site_valor_inconsistente' AS checagem,
         'valor_total diferente de subtotal + frete (contrato)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(valor_inconsistente) FROM horizonte.silver.site_pedidos) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_pedidos' AS tabela, 'site_subtotal_diferente_itens' AS checagem,
         'subtotal diferente da soma dos itens (contrato)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (WITH it AS (SELECT pedido_id, SUM(quantidade * preco_unitario) AS soma
            FROM horizonte.silver.site_pedido_itens WHERE item_atual GROUP BY pedido_id)
  SELECT COUNT(*) FROM horizonte.silver.site_pedidos p LEFT JOIN it USING (pedido_id)
  WHERE p.versao_atual AND (it.soma IS NULL OR it.soma <> p.subtotal)) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_pedido_itens' AS tabela, 'site_sku_sem_cadastro' AS checagem,
         'Item com SKU fora do catálogo (contrato, regra 4)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(sku_sem_cadastro) FROM horizonte.silver.site_pedido_itens) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_eventos' AS tabela, 'site_regra1_session_start' AS checagem,
         'Sessão sem session_start ou com evento anterior a ele (contrato, regra 1)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (WITH s AS (SELECT sessao_id, MIN(ocorrido_em) AS primeiro,
                   MIN(CASE WHEN evento = 'session_start' THEN ocorrido_em END) AS inicio
            FROM horizonte.silver.site_eventos WHERE valido GROUP BY sessao_id)
  SELECT COUNT(*) FROM s WHERE inicio IS NULL OR primeiro < inicio) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_eventos' AS tabela, 'site_regra2_um_purchase' AS checagem,
         'Pedido sem exatamente um purchase (contrato, regra 2)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (WITH c AS (SELECT pedido_id, COUNT(*) AS n FROM horizonte.silver.site_eventos
            WHERE valido AND evento = 'purchase' GROUP BY pedido_id)
  SELECT COUNT(*) FROM horizonte.silver.site_pedidos p LEFT JOIN c USING (pedido_id)
  WHERE p.versao_atual AND COALESCE(c.n, 0) <> 1) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_eventos' AS tabela, 'site_regra2_valor_purchase' AS checagem,
         'Valor do purchase diferente do valor_total do pedido (contrato, regra 2)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT(*) FROM horizonte.silver.site_eventos e
  JOIN horizonte.silver.site_pedidos p ON p.pedido_id = e.pedido_id AND p.versao_atual
  WHERE e.valido AND e.evento = 'purchase' AND e.valor <> p.valor_total) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_eventos' AS tabela, 'site_regra3_ordem_funil' AS checagem,
         'Compra sem add_to_cart e begin_checkout antes, na mesma sessão (contrato, regra 3)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (WITH s AS (SELECT sessao_id,
                   MIN(CASE WHEN evento = 'add_to_cart'    THEN ocorrido_em END) AS carrinho,
                   MIN(CASE WHEN evento = 'begin_checkout' THEN ocorrido_em END) AS checkout,
                   MIN(CASE WHEN evento = 'purchase'       THEN ocorrido_em END) AS compra
            FROM horizonte.silver.site_eventos WHERE valido GROUP BY sessao_id)
  SELECT COUNT(*) FROM s
  WHERE compra IS NOT NULL
    AND (carrinho IS NULL OR checkout IS NULL OR carrinho > compra OR checkout > compra)) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_eventos' AS tabela, 'site_eventos_reenviados' AS checagem,
         'Eventos repetidos por reenvio da exportação (informativo)' AS descricao,
         CAST(NULL AS BIGINT) AS esperado,
         (SELECT COUNT_IF(duplicada) FROM horizonte.silver.site_eventos) AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_pedidos' AS tabela, 'site_versoes_anteriores' AS checagem,
         'Versões antigas de pedidos, como os cancelamentos (informativo)' AS descricao,
         CAST(NULL AS BIGINT) AS esperado,
         (SELECT COUNT_IF(NOT versao_atual) FROM horizonte.silver.site_pedidos) AS encontrado
    UNION ALL
  SELECT 'silver' AS camada, 'site_avaliacoes' AS tabela, 'avaliacao_reenviada' AS checagem,
         'Sujeira plantada: avaliações repetidas (D25)' AS descricao,
         CAST(26 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(NOT versao_atual) FROM horizonte.silver.site_avaliacoes WHERE origem = 'SIMULADO') AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_avaliacoes' AS tabela, 'avaliacao_nota_invalida' AS checagem,
         'Sujeira plantada: nota fora de 1 a 5 (D25)' AS descricao,
         CAST(13 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(nota_invalida) FROM horizonte.silver.site_avaliacoes WHERE origem = 'SIMULADO') AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_avaliacoes' AS tabela, 'avaliacao_sku_sem_cadastro' AS checagem,
         'Sujeira plantada: SKU fora do catálogo (D25)' AS descricao,
         CAST(13 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(sku_sem_cadastro) FROM horizonte.silver.site_avaliacoes WHERE origem = 'SIMULADO') AS encontrado
  UNION ALL
  SELECT 'silver' AS camada, 'site_avaliacoes' AS tabela, 'avaliacao_sem_compra' AS checagem,
         'Avaliação sem compra anterior do mesmo cliente (só as de SKU sem cadastro, D25)' AS descricao,
         CAST(13 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(sem_compra) FROM horizonte.silver.site_avaliacoes WHERE origem = 'SIMULADO') AS encontrado       
  UNION ALL
  SELECT 'silver' AS camada, 'fato_vendas' AS tabela, 'fato_validas_sem_grupo' AS checagem,
         'Venda válida sem categoria ou grupo (perderia o vínculo com o catálogo)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(valido AND grupo IS NULL) FROM horizonte.silver.fato_vendas) AS encontrado
  UNION ALL
  SELECT 'gold' AS camada, 'gold_vendas_diarias' AS tabela, 'gold_receita_diferente_fato' AS checagem,
         'Canais em que a receita da gold difere da fato (agregação perdeu ou duplicou venda)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (WITH f AS (SELECT canal, SUM(receita) AS r FROM horizonte.silver.fato_vendas WHERE valido GROUP BY canal),
       g AS (SELECT canal, SUM(receita) AS r FROM horizonte.gold.gold_vendas_diarias GROUP BY canal)
  SELECT COUNT(*) FROM f FULL OUTER JOIN g USING (canal) WHERE f.r IS DISTINCT FROM g.r) AS encontrado
  UNION ALL
  SELECT 'gold' AS camada, 'gold_pedidos_diarios' AS tabela, 'gold_pedidos_diferente_fato' AS checagem,
         'Canais em que o número de pedidos da gold difere da fato' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (WITH f AS (SELECT canal, COUNT(DISTINCT pedido_id) AS n FROM horizonte.silver.fato_vendas WHERE valido GROUP BY canal),
       g AS (SELECT canal, SUM(pedidos) AS n FROM horizonte.gold.gold_pedidos_diarios GROUP BY canal)
  SELECT COUNT(*) FROM f FULL OUTER JOIN g USING (canal) WHERE f.n IS DISTINCT FROM g.n) AS encontrado
  UNION ALL
  SELECT 'gold' AS camada, 'gold_estoque_diario' AS tabela, 'gold_estoque_diferente_vendas' AS checagem,
         'Diferença entre saídas do estoque e itens vendidos no ERP, sem duplicadas' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT ABS((SELECT SUM(unidades_vendidas) FROM horizonte.gold.gold_estoque_diario)
            - (SELECT SUM(ABS(quantidade)) FROM horizonte.silver.erp_vendas WHERE NOT duplicada))) AS encontrado
  UNION ALL
  SELECT 'gold' AS camada, 'gold_catalogo' AS tabela, 'gold_catalogo_sku_repetido' AS checagem,
         'SKU repetido no catálogo do agente' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT(*) - COUNT(DISTINCT sku) FROM horizonte.gold.gold_catalogo) AS encontrado
  UNION ALL
  SELECT 'gold' AS camada, 'gold_catalogo' AS tabela, 'gold_catalogo_sem_descricao' AS checagem,
         'Produto sem descrição ou ocasiões (o agente não conseguiria recomendar)' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT COUNT_IF(descricao IS NULL OR ocasioes IS NULL) FROM horizonte.gold.gold_catalogo) AS encontrado
  UNION ALL
  SELECT 'gold' AS camada, 'gold_catalogo' AS tabela, 'gold_catalogo_avaliacoes_diferente' AS checagem,
         'Avaliações contadas no catálogo diferente da gold_avaliacoes' AS descricao,
         CAST(0 AS BIGINT) AS esperado,
         (SELECT ABS((SELECT SUM(qtd_avaliacoes) FROM horizonte.gold.gold_catalogo)
                   - (SELECT COUNT(*) FROM horizonte.gold.gold_avaliacoes))) AS encontrado
)
SELECT date_format(current_timestamp(), 'yyyyMMddHHmmss') AS execucao_id,
       current_timestamp()                                AS executado_em,
       camada, tabela, checagem, descricao, esperado, CAST(encontrado AS BIGINT) AS encontrado,
       CASE WHEN esperado IS NULL       THEN 'INFO'
            WHEN encontrado = esperado  THEN 'OK'
            ELSE 'FALHA' END                              AS status
FROM checagens;

### Última execução
Uma *view* sempre aponta para a execução mais recente: é o que um dashboard ou um alerta consulta.

In [0]:
CREATE OR REPLACE VIEW horizonte.gold.dq_ultima_execucao AS
SELECT *
FROM horizonte.gold.dq_resultados
WHERE execucao_id = (SELECT MAX(execucao_id) FROM horizonte.gold.dq_resultados);

In [0]:
SELECT status, checagem, esperado, encontrado, descricao
FROM horizonte.gold.dq_ultima_execucao
ORDER BY CASE status WHEN 'FALHA' THEN 1 WHEN 'INFO' THEN 3 ELSE 2 END, camada, checagem;
-- esperado: 18 OK, 2 INFO (0 e 0 no histórico), nenhuma FALHA